# Airbnb Intelligence — Model 02: Price Prediction

**Task:** predict `base_price_npr` using property characteristics, amenities and stay timing.

**Data provenance:** the source price observations identify themselves as `Synthetic Sample Data`. This experiment is useful for demonstrating a reproducible ML workflow, but it is not evidence of live Kathmandu Airbnb pricing.


In [ ]:
from pathlib import Path
import pandas as pd
import subprocess
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data" / "processed").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from within the project directory.")

DATA = PROJECT_ROOT / "data" / "processed" / "price_observations_clean.csv"
df = pd.read_csv(DATA)
print("Rows:", len(df))
print("Unique properties:", df.property_id.nunique())
display(df.head())


## 1. Target distribution and feature preparation

In [ ]:
display(df["base_price_npr"].describe())
display(df.groupby("property_type")["base_price_npr"].agg(["count", "mean", "median"]).sort_values("median", ascending=False))


## 2. Model comparison

The project training script fits a Gradient Boosting model and compares it with a median-price baseline. A group-based split holds out entire properties so that a property cannot appear in both train and test data.


In [ ]:
subprocess.run(
    [sys.executable, str(PROJECT_ROOT / "src" / "train_price_model.py")],
    cwd=PROJECT_ROOT,
    check=True,
)
metrics = pd.read_csv(PROJECT_ROOT / "reports" / "price_model_metrics.csv")
display(metrics)


## 3. Evaluation interpretation

- MAE: average absolute error in NPR; lower is better.
- RMSE: penalizes larger errors more strongly; lower is better.
- R²: variance explained on the held-out set; closer to 1 is generally better.
- The Gradient Boosting model is evaluated against a median-price baseline.
- The split is grouped by property_id to measure generalization to unseen properties.


In [ ]:
pred = pd.read_csv(PROJECT_ROOT / "reports" / "price_predictions.csv")
display(pred.head(10))
print("Mean absolute error (NPR):", round(pred.absolute_error_npr.mean(), 2))


## 4. Limitations

The dataset has 1,000 synthetic price observations from 400 property IDs. A grouped test split is more conservative than a row-random split, but the sample is still small and synthetic. No claim of real market accuracy should be made. Before deployment, replace or supplement it with permitted, timestamped, verifiable price observations and evaluate on a later time period.
